# llama.cpp 1.0 정리: 1.0이 뜻하는 것(버전 규칙·API 안정성), 설치 경로(Homebrew·winget·Docker 등)와 로컬 모델 첫 실행

위에서부터 차례로 실행해 보세요.

### crane 으로 공식 이미지 받아 풀기 (Docker 없는 환경용)

In [ ]:
%%bash
set -eo pipefail
mkdir -p /content/img && cd /content/img
curl -sL https://github.com/google/go-containerregistry/releases/latest/download/go-containerregistry_Linux_x86_64.tar.gz | tar xz crane
T=v1.0.0-b12359-b1b770d
mkdir -p light server
./crane export ghcr.io/ggml-org/llama.cpp:light-$T - | tar x -C light
./crane export ghcr.io/ggml-org/llama.cpp:server-$T - | tar x -C server
apt-get update -qq && apt-get install -y patchelf
for d in light server; do R=/content/img/$d; for b in $R/app/llama-*; do case $b in *.so*) ;; *) patchelf --set-interpreter $R/lib64/ld-linux-x86-64.so.2 $b;; esac; done; done
ls light/app | grep -v '\.so'

### --version 으로 버전 확인

In [ ]:
%%bash
R=/content/img/light
LD_LIBRARY_PATH=$R/app:$R/usr/lib/x86_64-linux-gnu:$R/lib/x86_64-linux-gnu $R/app/llama-cli --version 2>&1 | tail -3

### llama-cli 로 -hf 모델 받아 실행

In [ ]:
%%bash
R=/content/img/light
export HOME=/content SSL_CERT_FILE=/etc/ssl/certs/ca-certificates.crt
LD_LIBRARY_PATH=$R/app:$R/usr/lib/x86_64-linux-gnu:$R/lib/x86_64-linux-gnu $R/app/llama-cli -hf ggml-org/Qwen3.5-0.8B-GGUF:Q4_0 -p 'Say hello in one sentence.' -n 100 -st --reasoning off 2>&1 | grep -vE '^\s*$' | tail -12
LD_LIBRARY_PATH=$R/app:$R/usr/lib/x86_64-linux-gnu:$R/lib/x86_64-linux-gnu $R/app/llama-cli -cl

### llama-server 를 백그라운드로 띄우기

In [ ]:
%%bash
R=/content/img/server
export HOME=/content SSL_CERT_FILE=/etc/ssl/certs/ca-certificates.crt
LD_LIBRARY_PATH=$R/app:$R/usr/lib/x86_64-linux-gnu:$R/lib/x86_64-linux-gnu nohup $R/app/llama-server -hf ggml-org/Qwen3.5-0.8B-GGUF:Q4_0 --reasoning off > /content/server.log 2>&1 &
for i in $(seq 1 60); do curl -sf http://127.0.0.1:9931/health && break; sleep 2; done
grep listening /content/server.log

### curl 로 /v1/chat/completions 호출

In [ ]:
%%bash
curl -s http://localhost:9931/v1/models | jq -c '.data[0]|{id,object}'
curl -s http://localhost:9931/v1/chat/completions -H 'Content-Type: application/json' -H 'Authorization: Bearer no-key' -d '{"model":"gpt-3.5-turbo","messages":[{"role":"user","content":"Say hello in one sentence."}]}' | jq .

### 웹 UI 캡처용 Playwright 설치 (Chromium 포함)

In [ ]:
%%bash
set -eo pipefail
cd /content
pip install -q playwright
playwright install --with-deps chromium 2>&1 | tail -3

### 웹 UI 입력·전송·캡처 스크립트 만들기

In [ ]:
%%writefile /content/ui.py
from playwright.sync_api import sync_playwright

with sync_playwright() as pw:
    b = pw.chromium.launch(args=['--no-sandbox'])
    p = b.new_page(viewport={'width': 1280, 'height': 800})
    p.goto('http://localhost:9931/')
    p.wait_for_timeout(2500)
    p.screenshot(path='/content/outputs/webui_home.png')
    t = p.locator('.cm-content').first
    t.click()
    p.keyboard.type('Say hello in one sentence.')
    p.wait_for_timeout(500)
    p.screenshot(path='/content/outputs/webui_input.png')
    t.press('Enter')
    p.wait_for_timeout(9000)
    p.screenshot(path='/content/outputs/webui_chat.png')
    print(p.inner_text('body')[:400])
    b.close()

### 브라우저로 llama-server 웹 UI 접속해 채팅 화면 캡처

In [ ]:
%%bash
set -eo pipefail
mkdir -p /content/outputs
curl -s -o /dev/null -w "HTTP %{http_code}\n" http://localhost:9931/
python3 /content/ui.py
ls -l /content/outputs/webui_*.png

### 캡처한 웹 UI 화면 확인

In [ ]:
from IPython.display import Image, display
for f in ['webui_home', 'webui_input', 'webui_chat']:
    display(Image(f'/content/outputs/{f}.png', width=700))